In [3]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.8.0+cu129


In [4]:
import torch
import random
import numpy as np

def seed_torch(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
from dataclasses import dataclass

@dataclass
class ModelArgs:
    n_heads: int
    dim: int
    hidden_dim: int
    dropout: float
    max_seq_len: int
    n_layer: int
    vocab_size: int

In [6]:
import torch.nn as nn
import torch.nn.functional as F
import torch
import math

'''多头自注意力核心代码'''
class MultiHeadAttention(nn.Module):

    def __init__(self, args: ModelArgs, is_casual=False):

        # 初始化父类
        super().__init__()

        # 隐藏维度必须被头整除
        assert args.dim % args.n_heads == 0

        # 每个头的维度，等于模型维度除以头的总数。
        self.head_dim = args.dim // args.n_heads

        # 成员变量赋值
        self.n_heads = args.n_heads
        self.is_casual = is_casual

        # Wq, Wk, Wv变换矩阵，shape=[dim, dim], 注意：self.n_heads * self.head_dim=args.dim
        self.wq = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_heads * self.head_dim, bias=False)

        # 输出变换矩阵，shape=[dim, dim]
        self.wo = nn.Linear(self.n_heads * self.head_dim, args.dim, bias=False)

        # 注意力的dropout
        self.attn_dropout = nn.Dropout(args.dropout)

        # 残差的dropout
        self.res_dropout = nn.Dropout(args.dropout)

        # 掩码上三角矩阵，屏蔽未来的token
        if is_casual:
           mask = torch.full((1, 1, args.max_seq_len, args.max_seq_len), float("-inf"))
           mask = torch.triu(mask, diagonal=1)

           # 注册缓冲区
           self.register_buffer("mask", mask)


    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):

        # [batch_size, seq_len, dim]
        batch_size, seq_len, dim = q.shape

        # 计算Q,K,V，维度为 (batch_size, seq_len, dim) x (dim, dim) -> (batch_size, seq_len, dim)
        Q, K, V = self.wq(q), self.wk(k), self.wv(v)

        # 将 Q、K、V 拆分成多头，shape=(batch_size, seq_len, n_heads, head_dim)
        Q = Q.view(batch_size, seq_len, self.n_heads, self.head_dim)
        K = K.view(batch_size, seq_len, self.n_heads, self.head_dim)
        V = V.view(batch_size, seq_len, self.n_heads, self.head_dim)

        # 交换位置1和位置2，shape=(batch_size, n_heads, seq_len, head_dim)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)


        # 注意力计算
        # QK^T / sqrt(d_k)，(batch_size, n_heads, seq_len, head_dim) x (batch_size, n_heads, head_dim, seq_len) -> (batch_size, n_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(self.head_dim)

        # 计算掩码
        if self.is_casual:
            # 直接相加，-inf叠加的位置就等于mask掉了, max_seq_len可能大于seq_len
            scores = scores + self.mask[:, :, :seq_len, :seq_len]

        # 计算 softmax，shape=(batch_size, n_heads, seq_len, seq_len)
        scores = F.softmax(scores.float(), dim=-1).type_as(Q)

        # 计算dropout，维度不变
        scores = self.attn_dropout(scores)

        # 计算注意力加权输出，V * Score，维度为(batch_size, n_heads, seq_len, seq_len) x (batch_size, n_heads, seq_len, head_dim) -> (batch_size, n_heads, seq_len, head_dim)
        output = torch.matmul(scores, V)

        # 拼接多头注意力结果，(batch_size, n_heads, seq_len, head_dim) => (batch_size, seq_dim, dim)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # 输出层+残差
        output = self.wo(output)
        output = self.res_dropout(output)

        return output

In [7]:
class LayerNorm(nn.Module):
    """
    层归一化，用于对最后一个维度进行归一化。

    参数:
        feature_size: 输入特征的维度大小，即归一化的特征维度。
        epsilon: 防止除零的小常数。
    """

    def __init__(self, dim, eps=1e-6):
    	super().__init__()
    	self.gamma = nn.Parameter(torch.ones(dim))  # 可学习缩放参数，初始值为 1
    	self.beta = nn.Parameter(torch.zeros(dim))  # 可学习偏移参数，初始值为 0
    	self.eps = eps

    def forward(self, x):
    	# 计算均值和方差
    	mean = x.mean(-1, keepdim=True) # mean: [batch, max_len, 1]
    	std = x.std(-1, keepdim=True)   # std: [batch, max_len, 1]

    	return self.gamma * (x - mean) / (std + self.eps) + self.beta

In [8]:
class FFN(nn.Module):
    '''前馈神经网络'''
    def __init__(self, dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        # 第一个FC，从输入到隐藏层 从T5 开始，很多模型在FFN层都不用偏置了。
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)

        # 第二个FC，从隐藏层到输入
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)

        # dropout防止过拟合
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 前向传播函数
        # 首先，输入x通过第一层线性变换和RELU激活函数
        # 最后，通过第二层线性变换和dropout层
        return self.dropout(self.w2(F.relu(self.w1(x))))

In [9]:
class EncoderLayer(nn.Module):
    '''Encoder层'''
    def __init__(self, args):
        super().__init__()

        # 两个 LayerNorm，分别在 Attention 之前和 FFN 之前
        self.attention_norm = LayerNorm(args.dim)

        # Encoder不需要掩码
        self.attention = MultiHeadAttention(args, is_casual=False)

        self.fnn_norm = LayerNorm(args.dim)

        self.feed_forward = FFN(args.dim, args.hidden_dim, args.dropout)

    def forward(self, x):
        # 层归一化
        norm_x = self.attention_norm(x)

        # 多头自注意力
        h = x + self.attention.forward(norm_x, norm_x, norm_x)

        # 前馈神经网络
        out = h + self.feed_forward.forward(self.fnn_norm(h))

        return out

In [10]:
class Encoder(nn.Module):
    '''Encoder 块'''
    def __init__(self, args):
        super(Encoder, self).__init__()
        #  N 个 Encoder Layer叠加
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layer)])
        self.norm = LayerNorm(args.dim)

    def forward(self, x):
        "分别通过 N 层 Encoder Layer"
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

In [11]:
class DecoderLayer(nn.Module):
    '''解码层'''
    def __init__(self, args):
        super().__init__()
        # 解码层有三个LayerNorm，分别在 Mask Attention 之前、Self Attention 之前和 FFN 之前
        self.attention_mask_norm = LayerNorm(args.dim)

        # Decoder 的第一个部分是 Mask Attention，传入 is_casual=True
        self.mask_attention = MultiHeadAttention(args, is_casual=True)

        self.attention_norm = LayerNorm(args.dim)
        # Decoder 的第二个部分是 类似于 Encoder 的 Attention，传入 is_casual=False

        self.cross_attention = MultiHeadAttention(args, is_casual=False)
        self.ffn_norm = LayerNorm(args.dim)

        # 第三个部分是 FFN
        self.feed_forward = FFN(args.dim, args.hidden_dim, args.dropout)

    def forward(self, x, enc_out):
        # 层归一化
        norm_x = self.attention_mask_norm(x)

        # 掩码自注意力
        x = x + self.mask_attention.forward(norm_x, norm_x, norm_x)

        # 多头注意力
        norm_x = self.attention_norm(x)
        h = x + self.cross_attention.forward(norm_x, enc_out, enc_out)

        # 前馈神经网络
        out = h + self.feed_forward.forward(self.ffn_norm(h))

        return out

In [12]:
class Decoder(nn.Module):
    '''解码器'''
    def __init__(self, args):
        super(Decoder, self).__init__()

        # 一个 Decoder 由 N 个 Decoder Layer 组成
        self.layers = nn.ModuleList([DecoderLayer(args) for _ in range(args.n_layer)])
        self.norm = LayerNorm(args.dim)

    def forward(self, x, enc_out):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, enc_out)
        return self.norm(x)

In [13]:
class PositionEncoding(nn.Module):
    '''位置编码'''

    def __init__(self, args):
        super(PositionEncoding, self).__init__()

        # 序列的最大长度为max_seq_len
        pe = torch.zeros(args.max_seq_len, args.dim)
        position = torch.arange(0, args.max_seq_len).unsqueeze(1)

        # 计算转角 theta， 为什么要log？=》计算稳定性
        '''
            pe(pos, 2i) = sin(pos/(10000^(2i/dim)))
            pe(pos, 2i+1) = cos(pos/(10000^(2i/dim)))
        '''
        factor = torch.exp(
            torch.arange(0, args.dim, 2) * -(math.log(10000.0) / args.dim)
        )

        # 分别计算 sin 和 cos
        pe[:, 0::2] = torch.sin(position * factor)
        pe[:, 1::2] = torch.cos(position * factor)

        # 增加一个batch size的维度，对所有样本的位置编码都是一样的,[1, max_seq_len, dim]
        pe = pe.unsqueeze(0)

        # 注册到buffer，不计算梯度
        self.register_buffer("pe", pe)

    def forward(self, x):

        # 位置编码+词向量
        x = x + self.pe[:, :x.size(1)]
        return x


Transformer&Loss

In [14]:
#整体模型
class Transformer(nn.Module):
    def __init__(self,args):
        super().__init__()

        self.args = args
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(args.vocab_size,args.dim),
            wpe = PositionEncoding(args),
            drop = nn.Dropout(args.dropout),
            encoder = Encoder(args),
            decoder = Decoder(args)
        ) )

        # 最后的线性层，输入是dim，输出是vocab_size
        self.lm_head = nn.Linear(args.dim,args.vocab_size,bias=False)

    # 查看所有参数的数量
        print("number of parameters: %.2f M" % (self.get_num_params() / 1e6, ))
        '''统计所有参数的数量'''
    def get_num_params(self):
        return sum([p.numel() for p in self.parameters()])

    #前向计算函数
    def forward(self,src_input,tgt_input,targets=None):
        #输入的shape[batch,seq_len] (embedding之前的，没有dim）
        #targets shape [batch,seq_len]

        batch,seq_len = src_input.size()

        assert seq_len <= self.args.max_seq_len

        #Embedding层（共享） 输出:[batch,seq_len,dim]
        src_token_emb = self.transformer.wte(src_input)
        tgt_token_emb = self.transformer.wte(tgt_input)

        #位置编码，输出:[batch,seq_len,dim]  PositionEncoding层已经加过了
        src_pos_emb = self.transformer.wpe(src_token_emb)
        tgt_pos_emb = self.transformer.wpe(tgt_token_emb)

        # Dropout层
        src_pos_emb = self.transformer.drop(src_pos_emb)
        tgt_pos_emb = self.transformer.drop(tgt_pos_emb)

        #Encoder , 输出:[batch,seq_len,dim]
        enc_out  = self.transformer.encoder(src_pos_emb)

        #Decoder 输出：[batch,seq_len,dim]
        dec_out = self.transformer.decoder(tgt_pos_emb, enc_out)

        if targets is not None:
            #计算logits shape = [batch, seq_len, vocab_size]
            logits = self.lm_head(dec_out)

            #计算交叉熵
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            # 推理阶段，只需要计算logits，loss为None
            # <bos> A, B, C,
            logits = self.lm_head(dec_out[:, [-1], :])
            loss = None
        return logits, loss



Case 1

In [15]:
args = ModelArgs(n_heads=8, dim=768, hidden_dim=768*4, dropout=0.1, max_seq_len=512, n_layer=6, vocab_size=5000)
print(args)

ModelArgs(n_heads=8, dim=768, hidden_dim=3072, dropout=0.1, max_seq_len=512, n_layer=6, vocab_size=5000)


In [16]:
special_vocabs = {
    "PAD": 0,
    "BOS": 1,
    "EOS": 2
}

seed_torch(42)
batch_size = 4

'''
src input: <BOS>, A, B, C, <EOS>
tgt input: <BOS>, D, E, F
tgt output: D, E, F, <EOS>
'''

# 特殊字符
bos = special_vocabs["BOS"] + torch.zeros(batch_size, 1, dtype=int)
eos = special_vocabs["EOS"] + torch.zeros(batch_size, 1, dtype=int)

#输入
src = torch.randint(len(special_vocabs), args.vocab_size, (batch_size, args.max_seq_len - 2))
tgt = torch.randint(len(special_vocabs), args.vocab_size, (batch_size, args.max_seq_len - 1))
src_input = torch.cat([bos, src, eos], axis=-1)
tgt_input = torch.cat([bos, tgt], axis=-1)

#输出
tgt_output = torch.cat([tgt, eos], axis=-1)

print("src input: ", src_input)
print("tgt input: ", tgt_input)
print("tgt output: ", tgt_output)

src input:  tensor([[   1, 3305, 4975,  ..., 2866, 3889,    2],
        [   1, 3952, 4644,  ..., 2856, 2981,    2],
        [   1, 4898, 3363,  ..., 2199, 4747,    2],
        [   1, 2035, 3092,  ..., 4408, 1299,    2]])
tgt input:  tensor([[   1,  556, 2724,  ...,  624, 1490, 2270],
        [   1, 1802, 4560,  ..., 2118, 4115, 3290],
        [   1, 2383, 2682,  ..., 4520,  378, 2135],
        [   1, 1173, 2700,  ..., 4604,  657,  972]])
tgt output:  tensor([[ 556, 2724,  989,  ..., 1490, 2270,    2],
        [1802, 4560,   51,  ..., 4115, 3290,    2],
        [2383, 2682,  346,  ...,  378, 2135,    2],
        [1173, 2700, 3232,  ...,  657,  972,    2]])


In [17]:
# 模型
trans = Transformer(args)
print("模型结构: ", trans)

number of parameters: 106.82 M
模型结构:  Transformer(
  (transformer): ModuleDict(
    (wte): Embedding(5000, 768)
    (wpe): PositionEncoding()
    (drop): Dropout(p=0.1, inplace=False)
    (encoder): Encoder(
      (layers): ModuleList(
        (0-5): 6 x EncoderLayer(
          (attention_norm): LayerNorm()
          (attention): MultiHeadAttention(
            (wq): Linear(in_features=768, out_features=768, bias=False)
            (wk): Linear(in_features=768, out_features=768, bias=False)
            (wv): Linear(in_features=768, out_features=768, bias=False)
            (wo): Linear(in_features=768, out_features=768, bias=False)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (res_dropout): Dropout(p=0.1, inplace=False)
          )
          (fnn_norm): LayerNorm()
          (feed_forward): FFN(
            (w1): Linear(in_features=768, out_features=3072, bias=False)
            (w2): Linear(in_features=3072, out_features=768, bias=False)
            (dropout):

In [18]:
logits, loss = trans(src_input, tgt_input)
decode_ids = logits.argmax(dim=-1)
print("logits: ", logits.shape)
print("decode ids: ", decode_ids)

logits:  torch.Size([4, 1, 5000])
decode ids:  tensor([[4536],
        [ 338],
        [2843],
        [2387]])


In [19]:
logits, loss = trans(src_input, tgt_input, tgt_output)
print("logits: ", logits.shape, logits)
print("loss: ", loss)

logits:  torch.Size([4, 512, 5000]) tensor([[[ 0.1389,  0.4908,  0.2889,  ...,  1.3232, -0.7986, -0.6505],
         [-0.7657, -0.4415,  0.1996,  ..., -0.9714, -0.5153, -0.7088],
         [ 0.4274,  1.1833,  0.2273,  ..., -0.5622, -0.1804, -0.6670],
         ...,
         [ 0.0704, -0.0609,  0.5956,  ...,  0.3303,  0.2372, -0.1959],
         [-0.7723,  0.1343,  0.0906,  ...,  0.3571, -0.0242, -1.3760],
         [-0.7028, -0.1736, -0.3807,  ...,  0.0498,  0.3158, -0.8621]],

        [[-0.0337,  0.3171,  0.2116,  ...,  0.9230, -1.1465, -0.1172],
         [ 0.4076, -1.2820,  0.4329,  ..., -0.2321,  0.7357,  0.1516],
         [-0.5843,  0.5738,  0.2165,  ...,  0.1338,  0.2215, -1.1122],
         ...,
         [-0.5263,  0.0780,  0.2108,  ..., -0.4235,  0.6697,  0.0189],
         [-0.5303,  0.2499, -0.8228,  ...,  0.1980,  1.0195, -0.1597],
         [ 0.2181, -0.5880,  0.2069,  ..., -1.0405,  0.1594, -0.4218]],

        [[ 0.0037,  0.1146,  0.6808,  ...,  0.6266, -0.8337, -0.9026],
         